# Intrusion Detection System — Exploratory Data Analysis
### Dataset: CICIDS-2017

**Sections:**
1. Imports & Config
2. Data Loading
3. Shape & Basic Info
4. Missing & Infinite Values
5. Duplicate Rows
6. Label Distribution (Multi-class & Binary)
7. Class Imbalance
8. Descriptive Statistics
9. Per-column Variance
10. Skewness & Kurtosis
11. Correlation Heatmap
12. Feature Distributions (Top Skewed)

## 1. Imports & Config

In [ ]:
import os, zipfile, warnings, textwrap
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)
sns.set_theme(style='whitegrid', palette='tab10')
plt.rcParams['figure.dpi'] = 110
plt.rcParams['savefig.bbox'] = 'tight'

LABEL_COL = 'Label'

# ── persist all outputs to Google Drive so they survive runtime restarts ──
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/IDS_Project'   # central project folder on Drive
REPORT_DIR  = os.path.join(PROJECT_DIR, '01_eda')
FIGURES_DIR = os.path.join(REPORT_DIR, 'figures')
os.makedirs(FIGURES_DIR, exist_ok=True)

# ── helpers ──────────────────────────────────────────────────────────────────
_report_lines = []

def _log(text=''):
    _report_lines.append(str(text))

def _savefig(name, fig=None):
    path = os.path.join(FIGURES_DIR, name)
    (fig or plt).savefig(path, dpi=130, bbox_inches='tight')
    return path

def write_report():
    path = os.path.join(REPORT_DIR, 'EDA_Report.txt')
    with open(path, 'w', encoding='utf-8') as f:
        f.write('\n'.join(_report_lines))
    print(f'Report saved -> {path}')

print('Setup complete.')
print('  Project dir :', PROJECT_DIR)
print('  EDA outputs :', REPORT_DIR)

## 2. Mount Drive & Load Data
Mount Google Drive, extract the zip, then concatenate all 8 CSVs.

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Path to the zip on Drive (same location as the old notebook)
zip_path = '/content/drive/MyDrive/MachineLearningCSV.zip'

if not os.path.exists(zip_path):
    raise FileNotFoundError(f"File not found: {zip_path}")
else:
    print(f"Found: {zip_path}")

# Extract
extract_dir = '/content/cicids2017'
os.makedirs(extract_dir, exist_ok=True)
with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall(extract_dir)
print('Extraction complete.')

# Walk extracted dir and load all CSVs
dfs = []
for root, _, fnames in os.walk(extract_dir):
    for fname in sorted(fnames):
        if not fname.endswith('.csv'):
            continue
        fpath = os.path.join(root, fname)
        try:
            df_tmp = pd.read_csv(fpath, low_memory=False)
            df_tmp.columns = df_tmp.columns.str.strip()   # remove leading/trailing spaces
            df_tmp['source_file'] = fname.split('.')[0]
            dfs.append(df_tmp)
            print(f'  {fname:65s} -> {df_tmp.shape[0]:>7,} rows')
        except Exception as e:
            print(f'  SKIP {fname}: {e}')

df = pd.concat(dfs, ignore_index=True)
print(f'\nTotal combined shape: {df.shape}')
print(df[LABEL_COL].value_counts())

## 3. Shape & Basic Info

In [ ]:
print(f'Rows    : {df.shape[0]:,}')
print(f'Columns : {df.shape[1]:,}')
print(f'Feature columns (excl. Label & source_file): {df.shape[1] - 2}')

num_cols  = df.select_dtypes(include=[np.number]).columns.tolist()
feat_cols = [c for c in num_cols if c != 'source_file']
print(f'Numeric feature columns : {len(feat_cols)}')

_log('=' * 70)
_log('EDA REPORT  —  CICIDS-2017')
_log(f'Generated : {datetime.now().strftime("%Y-%m-%d %H:%M")}')
_log('=' * 70)
_log()
_log('── SECTION 1 : DATASET SHAPE ──────────────────────────────────────────')
_log(f'  Rows                      : {df.shape[0]:,}')
_log(f'  Columns (total)           : {df.shape[1]:,}')
_log(f'  Numeric feature columns   : {len(feat_cols)}')

In [ ]:
dtype_counts = df.dtypes.value_counts().rename(index=str)
rows_per_file = df.groupby('source_file').size().sort_values()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# dtype pie
axes[0].pie(dtype_counts, labels=dtype_counts.index, autopct='%1.0f%%',
            colors=sns.color_palette('Set2', len(dtype_counts)), startangle=90)
axes[0].set_title('Column dtype breakdown')

# rows per source file
rows_per_file.plot(kind='barh', ax=axes[1], color='steelblue')
axes[1].set_title('Rows per Source File')
axes[1].set_xlabel('Row count')
for bar in axes[1].patches:
    axes[1].text(bar.get_width() + 2000, bar.get_y() + bar.get_height() / 2,
                 f'{int(bar.get_width()):,}', va='center', fontsize=8)

plt.suptitle('Dataset Composition', fontsize=13, y=1.02)
plt.tight_layout()
_savefig('01_dataset_shape.png', fig)
plt.show()

print('Column dtypes:')
print(df.dtypes.to_string())

_log(f'  Rows per file:')
for fname, n in rows_per_file.items():
    _log(f'    {fname:<55} {n:>7,}')
_log(f'  dtype counts  : {dict(dtype_counts)}')

## 4. Missing & Infinite Values

In [ ]:
nan_counts = df.isnull().sum()
nan_pct    = df.isnull().mean() * 100

nan_df = pd.DataFrame({'NaN Count': nan_counts, 'NaN %': nan_pct})
nan_df = nan_df[nan_df['NaN Count'] > 0].sort_values('NaN Count', ascending=False)

_log()
_log('── SECTION 2 : MISSING & INFINITE VALUES ───────────────────────────────')

if nan_df.empty:
    print('No NaN values found in any column.')
    _log('  NaN values : NONE')
    # placeholder chart showing all-clean status
    fig, ax = plt.subplots(figsize=(6, 3))
    ax.text(0.5, 0.5, 'No Missing Values\nin any column',
            ha='center', va='center', fontsize=16, color='green',
            transform=ax.transAxes)
    ax.set_axis_off()
    ax.set_title('Missing Value Summary')
    _savefig('02a_nan_values.png', fig)
    plt.show()
else:
    print(f'Columns with NaNs: {len(nan_df)}')
    display(nan_df)
    _log(f'  Columns with NaN : {len(nan_df)}')
    _log(nan_df.to_string())

    fig, axes = plt.subplots(1, 2, figsize=(16, max(4, len(nan_df) * 0.4 + 1)))
    nan_df['NaN Count'].plot(kind='barh', ax=axes[0], color='tomato')
    axes[0].set_title('NaN Count per Column')
    axes[0].set_xlabel('NaN Count')
    nan_df['NaN %'].plot(kind='barh', ax=axes[1], color='salmon')
    axes[1].set_title('NaN % per Column')
    axes[1].set_xlabel('NaN %')
    plt.suptitle('Missing Values', fontsize=13)
    plt.tight_layout()
    _savefig('02a_nan_values.png', fig)
    plt.show()

# Heatmap of missingness pattern (sample 5k rows for speed)
sample = df.isnull().sample(min(5000, len(df)), random_state=42)
cols_with_nan = sample.columns[sample.any()].tolist()
if cols_with_nan:
    fig, ax = plt.subplots(figsize=(max(8, len(cols_with_nan)), 4))
    sns.heatmap(sample[cols_with_nan].T, cbar=False, ax=ax, cmap='Reds', yticklabels=True)
    ax.set_title('Missingness Pattern (sample 5 k rows)')
    ax.set_xlabel('Row index (sample)')
    _savefig('02b_nan_heatmap.png', fig)
    plt.show()
else:
    print('No NaN columns — missingness heatmap skipped.')

In [ ]:
inf_counts = {}
for col in feat_cols:
    n_inf = np.isinf(df[col]).sum()
    if n_inf > 0:
        inf_counts[col] = int(n_inf)

if not inf_counts:
    print('No infinite values found.')
    _log('  Inf values : NONE')
    fig, ax = plt.subplots(figsize=(6, 3))
    ax.text(0.5, 0.5, 'No Infinite Values Found', ha='center', va='center',
            fontsize=15, color='green', transform=ax.transAxes)
    ax.set_axis_off()
    ax.set_title('Infinite Value Summary')
    _savefig('02c_inf_values.png', fig)
    plt.show()
else:
    inf_series = pd.Series(inf_counts, name='Inf Count').sort_values(ascending=False)
    print(f'Columns with Inf values: {len(inf_series)}')
    display(inf_series.to_frame())
    _log(f'  Columns with Inf : {len(inf_series)}')
    _log(inf_series.to_string())

    fig, ax = plt.subplots(figsize=(10, max(3, len(inf_series) * 0.5)))
    inf_series.plot(kind='barh', ax=ax, color='darkorange')
    ax.set_title('Infinite Value Count per Column')
    ax.set_xlabel('Count of Inf values')
    for bar in ax.patches:
        ax.text(bar.get_width(), bar.get_y() + bar.get_height() / 2,
                f'  {int(bar.get_width()):,}', va='center', fontsize=8)
    plt.tight_layout()
    _savefig('02c_inf_values.png', fig)
    plt.show()

## 5. Duplicate Rows

In [ ]:
n_dups = df.duplicated().sum()
pct_dups = n_dups / len(df) * 100

print(f'Total duplicate rows : {n_dups:,}')
print(f'Duplicate percentage  : {pct_dups:.2f}%')

dup_by_label = (
    df[df.duplicated(keep=False)]
    .groupby(LABEL_COL)
    .size()
    .sort_values(ascending=False)
    .rename('Dup Count')
)

_log()
_log('── SECTION 3 : DUPLICATE ROWS ──────────────────────────────────────────')
_log(f'  Total duplicate rows : {n_dups:,}  ({pct_dups:.2f}%)')

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Donut chart: unique vs duplicate
sizes  = [len(df) - n_dups, n_dups]
labels = [f'Unique\n{len(df)-n_dups:,}', f'Duplicate\n{n_dups:,}']
colors = ['#4CAF50', '#F44336']
axes[0].pie(sizes, labels=labels, autopct='%1.2f%%', colors=colors,
            startangle=90, wedgeprops=dict(width=0.5))
axes[0].set_title('Unique vs Duplicate Rows')

# Duplicate counts per label
if not dup_by_label.empty:
    dup_by_label.plot(kind='barh', ax=axes[1], color='tomato')
    axes[1].set_title('Duplicate Rows per Label')
    axes[1].set_xlabel('Duplicate Count')
    for bar in axes[1].patches:
        axes[1].text(bar.get_width(), bar.get_y() + bar.get_height() / 2,
                     f'  {int(bar.get_width()):,}', va='center', fontsize=8)
    _log('  Duplicates by label:')
    _log(dup_by_label.to_string())
else:
    axes[1].text(0.5, 0.5, 'No Duplicates Found', ha='center', va='center',
                 fontsize=14, color='green', transform=axes[1].transAxes)
    axes[1].set_axis_off()
    axes[1].set_title('Duplicate Rows per Label')
    _log('  Duplicates by label : none')

plt.suptitle('Duplicate Row Analysis', fontsize=13)
plt.tight_layout()
_savefig('03_duplicates.png', fig)
plt.show()

## 6. Label Distribution
### 6a. Multi-class labels

In [ ]:
label_counts = df[LABEL_COL].value_counts()
label_pct    = df[LABEL_COL].value_counts(normalize=True) * 100

label_df = pd.DataFrame({'Count': label_counts, 'Percentage %': label_pct})
print(f'Unique labels: {len(label_df)}')
display(label_df)

_log()
_log('── SECTION 4 : LABEL DISTRIBUTION ─────────────────────────────────────')
_log(f'  Unique multi-class labels : {len(label_df)}')
_log(label_df.to_string())

In [ ]:
palette = sns.color_palette('tab10', len(label_counts))

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Bar
label_counts.plot(kind='bar', ax=axes[0], color=palette, edgecolor='white')
axes[0].set_title('Multi-class Label Counts', fontsize=12)
axes[0].set_xlabel('Label')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=40)
for p in axes[0].patches:
    axes[0].annotate(f'{int(p.get_height()):,}',
                     (p.get_x() + p.get_width() / 2, p.get_height()),
                     ha='center', va='bottom', fontsize=7)

# Pie
axes[1].pie(label_counts, labels=label_counts.index, autopct='%1.1f%%',
            colors=palette, startangle=140, pctdistance=0.82)
axes[1].set_title('Multi-class Label Proportions', fontsize=12)

plt.suptitle('Multi-class Label Distribution', fontsize=14)
plt.tight_layout()
_savefig('04a_multiclass_labels.png', fig)
plt.show()

# Stacked bar per source file
pivot = (df.groupby(['source_file', LABEL_COL])
           .size()
           .unstack(fill_value=0))
pivot.plot(kind='bar', stacked=True, figsize=(16, 5),
           colormap='tab20', edgecolor='none')
plt.title('Label composition per source file')
plt.xlabel('Source File')
plt.ylabel('Row count')
plt.xticks(rotation=35, ha='right')
plt.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=7)
plt.tight_layout()
_savefig('04b_labels_per_file.png')
plt.show()

### 6b. Binary labels (BENIGN vs ATTACK)

In [ ]:
df['binary_label'] = df[LABEL_COL].apply(
    lambda x: 'BENIGN' if str(x).strip() == 'BENIGN' else 'ATTACK'
)
bin_counts = df['binary_label'].value_counts()
bin_pct    = df['binary_label'].value_counts(normalize=True) * 100

bin_df = pd.DataFrame({'Count': bin_counts, 'Percentage %': bin_pct})
display(bin_df)

_log()
_log('  Binary label distribution:')
_log(bin_df.to_string())

In [ ]:
colors = ['#2196F3', '#F44336']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Bar
bin_counts.plot(kind='bar', ax=axes[0], color=colors, edgecolor='white')
axes[0].set_title('Binary Label Counts')
axes[0].set_xlabel('Class')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=0)
for p in axes[0].patches:
    axes[0].annotate(f'{int(p.get_height()):,}',
                     (p.get_x() + p.get_width() / 2, p.get_height()),
                     ha='center', va='bottom', fontsize=11)

# Donut
axes[1].pie(bin_counts, labels=bin_counts.index, autopct='%1.2f%%',
            colors=colors, startangle=90, explode=[0, 0.05],
            wedgeprops=dict(width=0.55))
axes[1].set_title('Binary Label Proportions')

# Per-file binary breakdown
bin_pivot = (df.groupby(['source_file', 'binary_label'])
               .size()
               .unstack(fill_value=0))
bin_pivot.plot(kind='barh', stacked=True, ax=axes[2], color=colors, edgecolor='none')
axes[2].set_title('Binary Labels per Source File')
axes[2].set_xlabel('Row Count')
axes[2].legend(loc='lower right')

plt.suptitle('Binary Classification — BENIGN vs ATTACK', fontsize=13)
plt.tight_layout()
_savefig('04c_binary_labels.png', fig)
plt.show()

## 7. Class Imbalance Analysis

In [ ]:
majority        = bin_counts.max()
minority        = bin_counts.min()
imbalance_ratio = majority / minority
smallest        = label_counts.min()
ratios          = (label_counts / smallest).sort_values(ascending=False)

print(f'Majority class : {majority:,}')
print(f'Minority class : {minority:,}')
print(f'Imbalance ratio: {imbalance_ratio:.2f}:1')
display(ratios.to_frame(name='Ratio vs Smallest Class'))

_log()
_log('── SECTION 5 : CLASS IMBALANCE ─────────────────────────────────────────')
_log(f'  Binary imbalance ratio : {imbalance_ratio:.2f}:1')
_log('  Multi-class ratios vs smallest class:')
_log(ratios.to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 5))

# Linear scale
label_counts.sort_values(ascending=False).plot(
    kind='bar', ax=axes[0],
    color=sns.color_palette('tab10', len(label_counts)), edgecolor='white')
axes[0].set_title('Multi-class Counts (linear scale)')
axes[0].set_xlabel('Label')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=40)

# Log scale
label_counts.sort_values(ascending=False).plot(
    kind='bar', ax=axes[1],
    color=sns.color_palette('tab10', len(label_counts)), edgecolor='white', logy=True)
axes[1].set_title('Multi-class Counts (log scale)')
axes[1].set_xlabel('Label')
axes[1].set_ylabel('Count (log)')
axes[1].tick_params(axis='x', rotation=40)

plt.suptitle('Class Imbalance — Multi-class', fontsize=13)
plt.tight_layout()
_savefig('05a_imbalance_multiclass.png', fig)
plt.show()

# Ratio bar
fig, ax = plt.subplots(figsize=(14, 4))
ratios.sort_values(ascending=False).plot(kind='bar', ax=ax, color='mediumorchid', edgecolor='white')
ax.set_title('Imbalance Ratio vs Smallest Class')
ax.set_ylabel('Ratio')
ax.set_xlabel('Label')
ax.tick_params(axis='x', rotation=40)
ax.axhline(1, color='black', linewidth=0.8, linestyle='--')
for p in ax.patches:
    ax.annotate(f'{p.get_height():.0f}x',
                (p.get_x() + p.get_width() / 2, p.get_height()),
                ha='center', va='bottom', fontsize=7)
plt.tight_layout()
_savefig('05b_imbalance_ratio.png', fig)
plt.show()

## 8. Descriptive Statistics

In [ ]:
df_clean = df[feat_cols].replace([np.inf, -np.inf], np.nan)

desc = df_clean.describe().T
desc['range'] = desc['max'] - desc['min']
desc['cv']    = desc['std'] / desc['mean'].replace(0, np.nan)
print(f'Descriptive stats for {len(feat_cols)} numeric features:')
display(desc)

_log()
_log('── SECTION 6 : DESCRIPTIVE STATISTICS ──────────────────────────────────')
_log(desc[['mean', 'std', 'min', '50%', 'max', 'range', 'cv']].to_string())

# ── Plot: mean ± std for all features (normalised so they fit one chart)
from sklearn.preprocessing import MinMaxScaler
scaler    = MinMaxScaler()
desc_norm = pd.DataFrame(scaler.fit_transform(desc[['mean', 'std', 'min', 'max']]),
                         columns=['mean', 'std', 'min', 'max'], index=desc.index)

fig, axes = plt.subplots(2, 1, figsize=(20, 10))

# Mean with error bars (std)
axes[0].bar(range(len(desc_norm)), desc_norm['mean'], yerr=desc_norm['std'],
            color='steelblue', alpha=0.7, capsize=2, error_kw=dict(elinewidth=0.5))
axes[0].set_xticks(range(len(desc_norm)))
axes[0].set_xticklabels(desc_norm.index, rotation=90, fontsize=5)
axes[0].set_title('Normalised Mean ± Std per Feature')
axes[0].set_ylabel('Normalised value')

# Range bar
axes[1].bar(range(len(desc)), desc['range'].values, color='coral', alpha=0.8)
axes[1].set_xticks(range(len(desc)))
axes[1].set_xticklabels(desc.index, rotation=90, fontsize=5)
axes[1].set_title('Value Range (max − min) per Feature')
axes[1].set_ylabel('Range')

plt.tight_layout()
_savefig('06a_descriptive_stats.png', fig)
plt.show()

# ── Boxplots for top-30 features by std (log-transformed for readability)
top30 = desc['std'].sort_values(ascending=False).head(30).index.tolist()
sample_box = df_clean[top30].sample(min(20000, len(df_clean)), random_state=42)
log_sample = np.log1p(sample_box.clip(lower=0))

fig, ax = plt.subplots(figsize=(20, 6))
log_sample.boxplot(ax=ax, vert=True, patch_artist=True,
                   boxprops=dict(facecolor='lightblue', color='navy'),
                   medianprops=dict(color='red', linewidth=1.5),
                   whiskerprops=dict(color='navy'),
                   flierprops=dict(marker='.', markersize=1, alpha=0.3))
ax.set_title('Boxplots — Top 30 Features by Std Dev  (log1p scale, sample 20k)')
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=7)
ax.set_ylabel('log1p(value)')
plt.tight_layout()
_savefig('06b_boxplots_top30.png', fig)
plt.show()

## 9. Per-column Variance

In [ ]:
variances = df_clean.var().sort_values(ascending=False)

print('Top 20 highest-variance features:')
display(variances.head(20).to_frame(name='Variance'))
print('\nBottom 20 lowest-variance features:')
display(variances.tail(20).to_frame(name='Variance'))

zero_var_cols = variances[variances == 0].index.tolist()

_log()
_log('── SECTION 7 : PER-COLUMN VARIANCE ─────────────────────────────────────')
_log(f'  Zero-variance (constant) columns : {len(zero_var_cols)}')
if zero_var_cols:
    _log(f'  {zero_var_cols}')
_log('  Top 10 highest-variance features:')
_log(variances.head(10).to_string())

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(20, 10))

# All features log-variance bar
log_var = np.log1p(variances)
log_var.plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='none')
axes[0].set_title('Per-feature Variance (log1p scale)')
axes[0].set_ylabel('log1p(Variance)')
axes[0].tick_params(axis='x', rotation=90, labelsize=5)

# Top 20 vs Bottom 20 side-by-side
top20_var = variances.head(20)
bot20_var = variances.tail(20)
combined  = pd.concat([top20_var, bot20_var])
colors_var = ['#1976D2'] * 20 + ['#EF5350'] * 20
combined.plot(kind='bar', ax=axes[1], color=colors_var, edgecolor='none')
axes[1].set_title('Top 20 (blue) vs Bottom 20 (red) Features by Variance')
axes[1].set_ylabel('Variance')
axes[1].tick_params(axis='x', rotation=90, labelsize=7)

# legend
from matplotlib.patches import Patch
axes[1].legend(handles=[Patch(color='#1976D2', label='Top 20'),
                         Patch(color='#EF5350', label='Bottom 20')])

plt.tight_layout()
_savefig('07_variance.png', fig)
plt.show()

print(f'Zero-variance columns: {len(zero_var_cols)}')
if zero_var_cols:
    print(zero_var_cols)

## 10. Skewness & Kurtosis

In [ ]:
skewness = df_clean.skew().sort_values(ascending=False)
kurtosis = df_clean.kurt().sort_values(ascending=False)
sk_df    = pd.DataFrame({'Skewness': skewness, 'Kurtosis': kurtosis})

print('Top 15 most positively skewed:')
display(sk_df.sort_values('Skewness', ascending=False).head(15))
print('\nTop 15 most negatively skewed:')
display(sk_df.sort_values('Skewness').head(15))

n_high_skew  = (skewness.abs() > 1).sum()
n_xhigh_skew = (skewness.abs() > 2).sum()
print(f'\n|skew| > 1 : {n_high_skew}  |  |skew| > 2 : {n_xhigh_skew}')

_log()
_log('── SECTION 8 : SKEWNESS & KURTOSIS ─────────────────────────────────────')
_log(f'  Features with |skew| > 1  : {n_high_skew}')
_log(f'  Features with |skew| > 2  : {n_xhigh_skew}')
_log('  Top 10 most skewed (absolute):')
_log(sk_df.reindex(skewness.abs().sort_values(ascending=False).head(10).index).to_string())

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(20, 12))

# Full skewness bar
skewness.plot(kind='bar', ax=axes[0, 0], color='coral', edgecolor='none')
axes[0, 0].axhline(0, color='black', linewidth=0.6)
axes[0, 0].axhline(1,  color='red', linewidth=0.8, linestyle='--', label='+1')
axes[0, 0].axhline(-1, color='red', linewidth=0.8, linestyle='--', label='-1')
axes[0, 0].set_title('Skewness per Feature (all)')
axes[0, 0].tick_params(axis='x', rotation=90, labelsize=5)
axes[0, 0].legend(fontsize=8)

# Full kurtosis bar
kurtosis.plot(kind='bar', ax=axes[0, 1], color='mediumpurple', edgecolor='none')
axes[0, 1].axhline(0, color='black', linewidth=0.6)
axes[0, 1].set_title('Kurtosis per Feature (all)')
axes[0, 1].tick_params(axis='x', rotation=90, labelsize=5)

# Top 20 skewed scatter (skewness vs kurtosis)
top20_sk = skewness.abs().sort_values(ascending=False).head(20).index
axes[1, 0].scatter(skewness[top20_sk], kurtosis[top20_sk],
                   c=range(len(top20_sk)), cmap='plasma', s=80, zorder=3)
for feat in top20_sk:
    axes[1, 0].annotate(feat, (skewness[feat], kurtosis[feat]),
                         fontsize=6, alpha=0.8)
axes[1, 0].axvline(0, color='gray', linewidth=0.6)
axes[1, 0].axhline(0, color='gray', linewidth=0.6)
axes[1, 0].set_title('Skewness vs Kurtosis — Top 20 skewed features')
axes[1, 0].set_xlabel('Skewness')
axes[1, 0].set_ylabel('Kurtosis')

# Distribution of skewness values (histogram)
axes[1, 1].hist(skewness, bins=30, color='coral', edgecolor='white', alpha=0.85)
axes[1, 1].axvline(1,  color='red', linestyle='--', label='+1')
axes[1, 1].axvline(-1, color='red', linestyle='--', label='-1')
axes[1, 1].set_title('Distribution of Skewness Values Across Features')
axes[1, 1].set_xlabel('Skewness')
axes[1, 1].set_ylabel('Number of features')
axes[1, 1].legend()

plt.suptitle('Skewness & Kurtosis Analysis', fontsize=14)
plt.tight_layout()
_savefig('08_skewness_kurtosis.png', fig)
plt.show()

## 11. Correlation Heatmap

In [ ]:
corr_matrix = df_clean.corr()

fig, ax = plt.subplots(figsize=(22, 18))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, cmap='coolwarm', center=0,
            vmin=-1, vmax=1, ax=ax, linewidths=0,
            xticklabels=True, yticklabels=True)
ax.set_title('Feature Correlation Heatmap (lower triangle)', fontsize=14)
ax.tick_params(axis='x', rotation=90, labelsize=4)
ax.tick_params(axis='y', labelsize=4)
plt.tight_layout()
_savefig('09a_correlation_heatmap.png', fig)
plt.show()

_log()
_log('── SECTION 9 : CORRELATION ─────────────────────────────────────────────')

In [ ]:
corr_upper = corr_matrix.where(mask == False)
high_corr_pairs = (
    corr_upper.stack()
    .reset_index()
    .rename(columns={'level_0': 'Feature A', 'level_1': 'Feature B', 0: 'Correlation'})
)
high_corr_pairs = high_corr_pairs[high_corr_pairs['Correlation'].abs() > 0.95].copy()
high_corr_pairs = high_corr_pairs.sort_values('Correlation', ascending=False, key=abs)

print(f'Highly correlated pairs (|r| > 0.95): {len(high_corr_pairs)}')
display(high_corr_pairs)

_log(f'  Highly correlated pairs (|r|>0.95) : {len(high_corr_pairs)}')
if not high_corr_pairs.empty:
    _log(high_corr_pairs.to_string())

# ── Bar chart of top correlated pairs
if not high_corr_pairs.empty:
    top_hc = high_corr_pairs.head(30).copy()
    top_hc['pair'] = top_hc['Feature A'].str[:18] + ' ↔ ' + top_hc['Feature B'].str[:18]

    fig, ax = plt.subplots(figsize=(14, max(5, len(top_hc) * 0.35)))
    colors_hc = ['#D32F2F' if v > 0 else '#1565C0' for v in top_hc['Correlation']]
    ax.barh(top_hc['pair'], top_hc['Correlation'], color=colors_hc)
    ax.axvline(0.95,  color='gray', linestyle='--', linewidth=0.8)
    ax.axvline(-0.95, color='gray', linestyle='--', linewidth=0.8)
    ax.set_title('Top Highly Correlated Feature Pairs (|r| > 0.95)')
    ax.set_xlabel('Pearson r')
    ax.tick_params(axis='y', labelsize=7)
    plt.tight_layout()
    _savefig('09b_high_corr_pairs.png', fig)
    plt.show()

# ── Correlation distribution histogram
fig, ax = plt.subplots(figsize=(10, 4))
flat_corr = corr_upper.stack().dropna()
ax.hist(flat_corr, bins=80, color='steelblue', edgecolor='none', alpha=0.8)
ax.axvline(0.95,  color='red',  linestyle='--', label='|r|=0.95')
ax.axvline(-0.95, color='red',  linestyle='--')
ax.set_title('Distribution of Pairwise Pearson Correlations')
ax.set_xlabel('Pearson r')
ax.set_ylabel('Pair count')
ax.legend()
plt.tight_layout()
_savefig('09c_corr_distribution.png', fig)
plt.show()

## 12. Feature Distributions (Top Skewed Features)

In [ ]:
top_skewed = skewness.abs().sort_values(ascending=False).head(12).index.tolist()

fig, axes = plt.subplots(3, 4, figsize=(22, 13))
axes = axes.flatten()

for i, col in enumerate(top_skewed):
    data = df_clean[col].dropna()
    cap  = data.quantile(0.99)
    data_capped = data[data <= cap]
    axes[i].hist(data_capped, bins=60, color='steelblue', edgecolor='none', alpha=0.85)
    axes[i].set_title(f'{col}\nskew={skewness[col]:.2f}  kurt={kurtosis[col]:.1f}', fontsize=8)
    axes[i].tick_params(labelsize=7)

plt.suptitle('Distributions — Top 12 Most Skewed Features (capped at 99th pct)', fontsize=13)
plt.tight_layout()
_savefig('10a_top_skewed_distributions.png', fig)
plt.show()

# Log-transformed versions of same features
fig, axes = plt.subplots(3, 4, figsize=(22, 13))
axes = axes.flatten()

for i, col in enumerate(top_skewed):
    data_log = np.log1p(df_clean[col].clip(lower=0).dropna())
    axes[i].hist(data_log, bins=60, color='teal', edgecolor='none', alpha=0.85)
    axes[i].set_title(f'log1p({col})', fontsize=8)
    axes[i].tick_params(labelsize=7)

plt.suptitle('Log1p-transformed Distributions — Top 12 Skewed Features', fontsize=13)
plt.tight_layout()
_savefig('10b_top_skewed_log_distributions.png', fig)
plt.show()

# KDE overlaid per class (binary) for top 6 skewed
top6 = top_skewed[:6]
fig, axes = plt.subplots(2, 3, figsize=(18, 8))
axes = axes.flatten()

for i, col in enumerate(top6):
    cap = df_clean[col].quantile(0.99)
    for cls, grp in df.groupby('binary_label'):
        vals = grp[col].clip(upper=cap).dropna()
        axes[i].hist(vals, bins=50, alpha=0.5, label=cls, density=True,
                     color='#2196F3' if cls == 'BENIGN' else '#F44336')
    axes[i].set_title(col, fontsize=8)
    axes[i].legend(fontsize=7)
    axes[i].tick_params(labelsize=7)

plt.suptitle('Feature Distributions — BENIGN vs ATTACK (top 6 skewed)', fontsize=13)
plt.tight_layout()
_savefig('10c_class_distributions.png', fig)
plt.show()

_log()
_log('── SECTION 10 : FEATURE DISTRIBUTIONS ──────────────────────────────────')
_log(f'  Top 12 skewed features analysed: {top_skewed}')

## 13. Summary & Key Findings

In [ ]:
_log()
_log('=' * 70)
_log('SUMMARY')
_log('=' * 70)
_log(f'Total rows              : {len(df):,}')
_log(f'Total columns           : {df.shape[1]}')
_log(f'Numeric feature columns : {len(feat_cols)}')
_log(f'Unique labels (multi)   : {df[LABEL_COL].nunique()}')
_log()
_log(f'NaN columns             : {0 if nan_df.empty else len(nan_df)}')
_log(f'Inf-value columns       : {len(inf_counts)}')
_log(f'Duplicate rows          : {n_dups:,}  ({n_dups/len(df)*100:.2f}%)')
_log(f'Zero-variance cols      : {len(zero_var_cols)}')
_log(f'Highly skewed (|s|>1)   : {n_high_skew}')
_log(f'Extremely skewed (|s|>2): {n_xhigh_skew}')
_log(f'High-corr pairs >0.95   : {len(high_corr_pairs)}')
_log()
_log('Binary class distribution:')
for cls, cnt in bin_counts.items():
    _log(f'  {cls:10s}: {cnt:>9,}  ({cnt/len(df)*100:.2f}%)')
_log()
_log('Recommended next steps:')
_log('  1. Drop duplicate rows')
_log('  2. Replace Inf values with NaN, then impute or drop')
_log('  3. Drop zero-variance columns')
_log('  4. Apply log1p transform to highly skewed features')
_log('  5. Address class imbalance (SMOTE on train set only)')
_log('  6. Remove one of each highly-correlated pair (|r|>0.95)')
_log('  7. StandardScaler / MinMaxScaler before modelling')
_log()
_log('Figures saved:')
FIGURE_INDEX = [
    ('01_dataset_shape.png',              'Dtype pie + rows per file'),
    ('02a_nan_values.png',                'NaN count & percentage per column'),
    ('02b_nan_heatmap.png',               'Missingness pattern heatmap'),
    ('02c_inf_values.png',                'Infinite value counts'),
    ('03_duplicates.png',                 'Unique vs duplicate rows + per-label breakdown'),
    ('04a_multiclass_labels.png',         'Multi-class bar + pie'),
    ('04b_labels_per_file.png',           'Stacked label composition per source file'),
    ('04c_binary_labels.png',             'Binary bar, donut, per-file breakdown'),
    ('05a_imbalance_multiclass.png',      'Linear vs log class counts'),
    ('05b_imbalance_ratio.png',           'Imbalance ratio vs smallest class'),
    ('06a_descriptive_stats.png',         'Normalised mean±std + value range'),
    ('06b_boxplots_top30.png',            'Boxplots top-30 features by std (log scale)'),
    ('07_variance.png',                   'Log-variance bar + top20 vs bottom20'),
    ('08_skewness_kurtosis.png',          'Skew bar, kurt bar, scatter, histogram'),
    ('09a_correlation_heatmap.png',       'Full lower-triangle correlation heatmap'),
    ('09b_high_corr_pairs.png',           'Highly correlated pairs bar'),
    ('09c_corr_distribution.png',         'Distribution of all pairwise correlations'),
    ('10a_top_skewed_distributions.png',  'Raw histograms top-12 skewed features'),
    ('10b_top_skewed_log_distributions.png', 'Log1p histograms same features'),
    ('10c_class_distributions.png',       'BENIGN vs ATTACK overlay top-6 features'),
]
for fname, desc in FIGURE_INDEX:
    _log(f'  {fname:<45} {desc}')

# Write the report text file
write_report()

# Print summary to notebook
print('=' * 65)
print('EDA SUMMARY')
print('=' * 65)
print(f'Rows            : {len(df):,}')
print(f'Columns         : {df.shape[1]}')
print(f'Numeric features: {len(feat_cols)}')
print(f'Unique labels   : {df[LABEL_COL].nunique()}')
print()
print(f'NaN columns     : {0 if nan_df.empty else len(nan_df)}')
print(f'Inf columns     : {len(inf_counts)}')
print(f'Duplicates      : {n_dups:,}  ({n_dups/len(df)*100:.2f}%)')
print(f'Zero-var cols   : {len(zero_var_cols)}')
print(f'|skew|>1 cols   : {n_high_skew}')
print(f'High-corr pairs : {len(high_corr_pairs)}')
print()
for cls, cnt in bin_counts.items():
    print(f'  {cls:10s}: {cnt:>9,}  ({cnt/len(df)*100:.2f}%)')
print()
print(f'Report  -> {REPORT_DIR}/EDA_Report.txt')
print(f'Figures -> {FIGURES_DIR}/')
print(f'  {len(FIGURE_INDEX)} figures saved')